# Weight Sampling & Sweep Methodology Audit

**Motivation.** Advisor (Apr 17) flagged that the reported "best λ assigns <2% to PDE" looks suspicious. Two methodology issues:

1. **Biased sweep metric.** `sweep_lambda` defaults to `exclude_terms={"pde"}` — the validation metric is `L_bc + L_ic + L_data` (PDE is EXCLUDED). Of course the best λ is then the one that let the network ignore PDE. Circular.
2. **Uniform cube sampling without normalisation.** Independent U(0,1) per component means most training samples have all weights ≈ 0.5 — no contrastive trade-offs. Total loss scale varies 0–k across steps, jittering the effective learning rate.

This notebook runs four audits:

- **§1 Fair sweep.** Re-sweep existing checkpoints with `exclude_terms=set()` (all terms in metric) and 10k candidates. Does the reported "best λ" change? Does rel-L2 vs reference change?
- **§2 Sweep reliability.** Compute best-λ variance across seeds and candidate counts — is the 500-candidate sweep noisy?
- **§3 λ-invariance plot.** Inference-time λ sweep → prediction envelope. Advisor's requested paper figure. Low envelope = network learned to be robust.
- **§4 `uniform_normalized` retraining (Burgers).** New sampler mode: U(0,1)^k then divide by sum, so samples lie on the simplex. Test whether normalising recovers / retains the 2× Burgers improvement from uniform.

Cells in §1–§3 only load existing checkpoints (fast, ~5 min total). §4 retrains (~75 min per equation, marked clearly).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import clear_output

from pinns.model import LossConditionalPINN
from pinns.lambda_sampler import LambdaSampler
from pinns.inference import sweep_lambda
from pinns.training import train_lc_pinn
from pinns.device import select_device, device_info

from pinns.equations import buckley_leverett as bl
from pinns.equations import burgers as burg

device = select_device()
print(f"Device: {device_info(device)}")
torch.manual_seed(0)
np.random.seed(0)

HIDDEN_DIMS = [64, 64, 64, 64]

/Users/anna/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps (Apple GPU)


## §1 Fair sweep — does the "tiny λ_pde" finding survive?

Re-sweep four existing trained models with two configurations:

| Config | `exclude_terms` | `n_candidates` |
|--------|----------------|----------------|
| **BIASED** (current default) | `{"pde"}` | 500 |
| **FAIR** | `set()` (all terms) | 10 000 |

Then evaluate rel-L2 vs reference at each reported best-λ. If the FAIR best-λ has noticeably higher λ_pde AND comparable rel-L2, the "<2% PDE" claim was a sweep artefact. If FAIR picks the same λ, the finding is robust.

Checkpoints audited:
- `bl_lr1e3_300k_mar22.pt` — BL logspace v1 (the reported BL best)
- `bl_lc_pinn_uniform.pt` — BL uniform (Apr 18)
- `burgers_lc_pinn.pt` — Burgers logspace
- `burgers_lc_pinn_uniform.pt` — Burgers uniform (Apr 17 star result)

In [2]:
# Reference solutions + training batches (needed for sweep loss + rel-L2 evaluation).
print("BL reference solution…")
bl_ref = bl.compute_reference_solution(nx=500)
bl_batch = bl.generate_training_data(bl_ref, n_pde=2000, n_bc=200, n_ic=200, n_data=200, device=device)

print("Burgers reference solution…")
burg_ref = burg.compute_reference_solution()
burg_batch = burg.generate_training_data(burg_ref, device=device)
print("Ready.")

BL reference solution…
Burgers reference solution…
Ready.


In [3]:
EXPERIMENTS = [
    dict(name="BL logspace (v1)",   eq=bl,   ckpt="bl_lr1e3_300k_mar22.pt",    mode="logspace", batch=bl_batch,   ref=bl_ref),
    dict(name="BL uniform",         eq=bl,   ckpt="bl_lc_pinn_uniform.pt",     mode="uniform",  batch=bl_batch,   ref=bl_ref),
    dict(name="Burgers logspace",   eq=burg, ckpt="burgers_lc_pinn.pt",        mode="logspace", batch=burg_batch, ref=burg_ref),
    dict(name="Burgers uniform",    eq=burg, ckpt="burgers_lc_pinn_uniform.pt",mode="uniform",  batch=burg_batch, ref=burg_ref),
]

def load_model(exp):
    eq = exp["eq"]
    model = LossConditionalPINN(eq.DIM_PHYS, eq.DIM_LAMBDA, HIDDEN_DIMS).to(device)
    ckpt = torch.load(f"../checkpoints/{exp['ckpt']}", map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    return model, ckpt["best_log_lambda"].to(device)

def mean_rel_l2(errors_dict):
    return float(np.mean(list(errors_dict.values())))

In [4]:
# Run both BIASED (exclude PDE, n=500) and FAIR (include all terms, n=10000) sweeps.

rows = []
for exp in EXPERIMENTS:
    print(f"\n=== {exp['name']} ===")
    model, ll_saved = load_model(exp)
    sampler = LambdaSampler(dim=exp["eq"].DIM_LAMBDA, device=device, mode=exp["mode"])

    # Saved best-λ (whatever the original notebook produced)
    saved_errors = exp["eq"].evaluate(model, ll_saved, exp["ref"], device)
    saved_rel = mean_rel_l2(saved_errors)

    # BIASED sweep (current default: exclude PDE, 500 candidates)
    torch.manual_seed(42)
    ll_biased, p_biased, _ = sweep_lambda(
        model, exp["batch"], sampler, device,
        loss_fn=exp["eq"].compute_losses,
        n_candidates=500, exclude_terms={"pde"},
    )
    biased_errors = exp["eq"].evaluate(model, ll_biased, exp["ref"], device)
    biased_rel = mean_rel_l2(biased_errors)

    # FAIR sweep (all terms, 10k candidates)
    torch.manual_seed(42)
    ll_fair, p_fair, _ = sweep_lambda(
        model, exp["batch"], sampler, device,
        loss_fn=exp["eq"].compute_losses,
        n_candidates=10_000, exclude_terms=set(),
    )
    fair_errors = exp["eq"].evaluate(model, ll_fair, exp["ref"], device)
    fair_rel = mean_rel_l2(fair_errors)

    rows.append(dict(
        name=exp["name"],
        saved_rel=saved_rel,
        biased_p=p_biased.cpu().numpy().round(4),
        biased_rel=biased_rel,
        fair_p=p_fair.cpu().numpy().round(4),
        fair_rel=fair_rel,
    ))

print("\n")
print(f"{'Model':<22}  {'saved rel-L2':>13}  {'biased rel-L2':>13}  {'fair rel-L2':>13}")
print("-" * 72)
for r in rows:
    print(f"{r['name']:<22}  {r['saved_rel']:>13.4f}  {r['biased_rel']:>13.4f}  {r['fair_rel']:>13.4f}")

print("\nBest-λ weights (biased vs fair):")
for r in rows:
    print(f"  {r['name']:<22}")
    print(f"    biased: {r['biased_p']}")
    print(f"    fair:   {r['fair_p']}")



=== BL logspace (v1) ===
Best log(lambda):     [-2.967  5.266  2.809  2.747]
Best weights (logspace): [2.000e-04 8.573e-01 7.340e-02 6.910e-02]
Best validation loss: 2.661322e-03
Best log(lambda):     [-0.575  5.214  0.841 -2.472]
Best weights (logspace): [3.000e-03 9.841e-01 1.240e-02 5.000e-04]
Best validation loss: 1.275438e+02

=== BL uniform ===
Best log(lambda):     [-4.13  -0.035 -7.383 -2.076]
Best weights (uniform): [1.610e-02 9.656e-01 6.000e-04 1.255e-01]
Best validation loss: 4.088569e-03
Best log(lambda):     [-2.121 -3.359 -2.381 -4.366]
Best weights (uniform): [0.12   0.0348 0.0925 0.0127]
Best validation loss: 7.534148e+01

=== Burgers logspace ===
Best log(lambda):     [-0.664  3.105  3.131  1.775]
Best weights (logspace): [0.01   0.4322 0.4435 0.1143]
Best validation loss: 2.057321e-07
Best log(lambda):     [ 0.318  0.128  0.377 -0.368]
Best weights (logspace): [0.2949 0.2439 0.3128 0.1485]
Best validation loss: 3.682497e-06

=== Burgers uniform ===
Best log(lambda):

### Interpreting §1

- If `biased_rel ≈ fair_rel` but the FAIR best-λ has higher λ_pde: the "<2% PDE" claim is a metric artefact. Both λ configurations give equivalent solution quality; we just happened to pick the one that hides the PDE weight.
- If `fair_rel < biased_rel`: the biased sweep was leaving quality on the table.
- If `fair_rel > biased_rel`: the excluded-PDE proxy actually correlates better with rel-L2 than the full loss does. Unlikely but possible.

Report whichever outcome occurs in EXPERIMENTS.md.

## §2 Sweep reliability — is the best-λ estimate stable?

500 candidates in a 4-D cube (volume 1) is sparse. Repeat the FAIR sweep 20 times with different seeds across three candidate counts — measure the spread of the reported best-λ and best rel-L2.

If the spread is small, the choice of `n_candidates` doesn't matter. If the spread is large at n=500 and shrinks at n=10000, our reported best-λ values are noisy and the FAIR configuration is the one to trust.

In [ ]:
TARGET = next(e for e in EXPERIMENTS if e["name"] == "Burgers uniform")  # star result
model, _ = load_model(TARGET)
sampler = LambdaSampler(dim=TARGET["eq"].DIM_LAMBDA, device=device, mode=TARGET["mode"])

def repeat_sweep(n_candidates, n_seeds=20):
    best_ps = []
    best_rels = []
    for seed in range(n_seeds):
        torch.manual_seed(seed)
        ll, p, _ = sweep_lambda(
            model, TARGET["batch"], sampler, device,
            loss_fn=TARGET["eq"].compute_losses,
            n_candidates=n_candidates, exclude_terms=set(),
        )
        errs = TARGET["eq"].evaluate(model, ll, TARGET["ref"], device)
        best_ps.append(p.cpu().numpy())
        best_rels.append(mean_rel_l2(errs))
    return np.array(best_ps), np.array(best_rels)

print("Burgers uniform — best-λ variance over seeds")
for n in (500, 2000, 10000):
    ps, rels = repeat_sweep(n, n_seeds=20)
    print(f"\nn_candidates={n}")
    print(f"  best-λ mean:  {ps.mean(0).round(3)}")
    print(f"  best-λ std:   {ps.std(0).round(3)}")
    print(f"  rel-L2 mean:  {rels.mean():.4f}   std: {rels.std():.4f}   min: {rels.min():.4f}   max: {rels.max():.4f}")

Burgers uniform — best-λ variance over seeds
Best log(lambda):     [-0.285 -0.146 -1.043 -1.495]
Best weights (uniform): [0.7519 0.864  0.3525 0.2242]
Best validation loss: 7.529946e-07
Best log(lambda):     [-0.365 -0.519 -1.018 -0.862]
Best weights (uniform): [0.6944 0.5949 0.3613 0.4224]
Best validation loss: 7.527985e-07
Best log(lambda):     [-0.288 -1.001 -0.593 -1.262]
Best weights (uniform): [0.7496 0.3674 0.5524 0.2831]
Best validation loss: 7.529044e-07
Best log(lambda):     [-0.321 -0.906 -0.626 -1.013]
Best weights (uniform): [0.7255 0.404  0.5345 0.3633]
Best validation loss: 7.527563e-07
Best log(lambda):     [-0.262 -0.666 -0.98  -1.118]
Best weights (uniform): [0.7692 0.5139 0.3752 0.327 ]
Best validation loss: 7.526972e-07
Best log(lambda):     [-0.308 -0.347 -1.178 -1.233]
Best weights (uniform): [0.7348 0.7065 0.308  0.2915]
Best validation loss: 7.528205e-07
Best log(lambda):     [-0.325 -0.564 -0.593 -1.442]
Best weights (uniform): [0.7222 0.5688 0.5529 0.2364]
Bes

## §3 λ-invariance plot (paper figure)

Advisor (Apr 17, line 80): *"ideally, the network learns to ignore the λ noise — prediction variance across λ should be small."* Plot prediction envelope at inference time across many λ draws.

For Burgers at t=0.75 (shock fully developed) and BL at t=0.3: sweep 200 λ vectors, predict, overlay predictions + reference, and report rel-L2 distribution.

In [ ]:
@torch.no_grad()
def invariance_plot(exp, t_val, n_lambdas=200, ax=None):
    model, _ = load_model(exp)
    sampler = LambdaSampler(dim=exp["eq"].DIM_LAMBDA, device=device, mode=exp["mode"])
    x_ref, u_ref = exp["ref"][t_val]

    torch.manual_seed(1)
    log_lams, _ = sampler.sample_batch(n=n_lambdas, step=8000)
    preds = []
    errs = []
    for i in range(n_lambdas):
        u_pred = exp["eq"].predict_solution(model, log_lams[i], x_ref, t_val, device)
        preds.append(u_pred)
        errs.append(float(np.linalg.norm(u_pred - u_ref) / (np.linalg.norm(u_ref) + 1e-10)))
    preds = np.stack(preds)
    errs = np.array(errs)

    ax.fill_between(x_ref, preds.min(0), preds.max(0), alpha=0.2, label=f'Envelope across {n_lambdas} λ')
    ax.plot(x_ref, preds.mean(0), 'b--', lw=1, label='Mean prediction')
    ax.plot(x_ref, u_ref, 'k-', lw=1.5, label='Reference')
    ax.set_title(f"{exp['name']}  t={t_val}\nrel-L2 over λ: mean={errs.mean():.4f}  std={errs.std():.4f}  max={errs.max():.4f}")
    ax.set_xlabel('x'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    return errs

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharey=False)
configs = [
    ("Burgers logspace", 0.75, axes[0, 0]),
    ("Burgers uniform",  0.75, axes[0, 1]),
    ("BL logspace (v1)", 0.3,  axes[1, 0]),
    ("BL uniform",       0.3,  axes[1, 1]),
]
for name, t_val, ax in configs:
    exp = next(e for e in EXPERIMENTS if e["name"] == name)
    invariance_plot(exp, t_val, ax=ax)

fig.suptitle('λ-invariance of LC-PINN predictions (tight envelope = robust)', y=1.02)
plt.tight_layout()
plt.savefig('../results/fig_lambda_invariance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: ../results/fig_lambda_invariance.png")

## §4 `uniform_normalized` retraining — Burgers

**New sampler mode:** `p ~ U(0,1)^k`, then `p ← p / sum(p)` so `Σpᵢ = 1`. Combines uniform's independence with simplex's contrastive structure. Addresses advisor's second concern (line 44–57 of transcript).

Retrain Burgers (200k steps, ~75 min on MPS) and compare against `uniform` (0.0004) and `logspace` (0.0009). **Training cell below is long-running — run it when you have time.**

In [ ]:
def make_live_plot_callback(title="Training"):
    def on_log(history):
        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.5))
        fig.suptitle(title)
        steps = history["step"]
        ax1.set_title("Loss components (log-log)")
        for key in history:
            if key in ("step", "total", "hw", "elapsed_sec"): continue
            ax1.plot(steps, history[key], label=key.upper(), alpha=0.7)
        ax1.plot(steps, history["total"], 'k-', lw=2, label="Total")
        ax1.set_xscale("log"); ax1.set_yscale("log")
        ax1.set_xlabel("Step"); ax1.set_ylabel("Loss")
        ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)
        ax2.set_title("Total loss (semilogy)")
        ax2.semilogy(steps, history["total"], 'k-', lw=1.5)
        ax2.set_xlabel("Step"); ax2.set_ylabel("Total loss"); ax2.grid(True, alpha=0.3)
        fig.tight_layout(); plt.show(); plt.close(fig)
    return on_log

torch.manual_seed(42)
np.random.seed(42)

model_unifnorm = LossConditionalPINN(burg.DIM_PHYS, burg.DIM_LAMBDA, HIDDEN_DIMS).to(device)
sampler_unifnorm = LambdaSampler(dim=burg.DIM_LAMBDA, device=device, mode="uniform_normalized")

history_unifnorm = train_lc_pinn(
    model_unifnorm, sampler_unifnorm, burg_batch, device,
    loss_fn=burg.compute_losses,
    n_epochs=200_000, lr=1e-3, log_every=5_000,
    on_log=make_live_plot_callback("LC-PINN Burgers uniform_normalized"),
)

# FAIR sweep (all terms, 10k candidates) so we're honest about the new mode too.
best_ll_un, best_p_un, _ = sweep_lambda(
    model_unifnorm, burg_batch, sampler_unifnorm, device,
    loss_fn=burg.compute_losses,
    n_candidates=10_000, exclude_terms=set(),
)
print(f"\nBest λ (uniform_normalized, simplex weights): {best_p_un.cpu().numpy().round(4)}")

torch.save({
    "model_state_dict": model_unifnorm.state_dict(),
    "best_log_lambda": best_ll_un,
}, "../checkpoints/burgers_lc_pinn_uniform_normalized.pt")

unifnorm_errors = burg.evaluate(model_unifnorm, best_ll_un, burg_ref, device)
print(f"rel-L2 per snapshot: {unifnorm_errors}")
print(f"Mean: {np.mean(list(unifnorm_errors.values())):.4f}")

## §4b `uniform_normalized` retraining — Buckley-Leverett (optional, ~96 min)

Does normalising uniform flip BL's result (uniform loses to logspace at 0.1433 vs 0.1252)? Expensive — run only if Burgers result is inconclusive.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

model_bl_un = LossConditionalPINN(bl.DIM_PHYS, bl.DIM_LAMBDA, HIDDEN_DIMS).to(device)
sampler_bl_un = LambdaSampler(dim=bl.DIM_LAMBDA, device=device, mode="uniform_normalized")

history_bl_un = train_lc_pinn(
    model_bl_un, sampler_bl_un, bl_batch, device,
    loss_fn=bl.compute_losses,
    n_epochs=300_000, lr=1e-3, log_every=5_000,
    on_log=make_live_plot_callback("LC-PINN BL uniform_normalized"),
)

best_ll_bl, best_p_bl, _ = sweep_lambda(
    model_bl_un, bl_batch, sampler_bl_un, device,
    loss_fn=bl.compute_losses,
    n_candidates=10_000, exclude_terms=set(),
)
print(f"\nBest λ (BL uniform_normalized): {best_p_bl.cpu().numpy().round(4)}")

torch.save({
    "model_state_dict": model_bl_un.state_dict(),
    "best_log_lambda": best_ll_bl,
}, "../checkpoints/bl_lc_pinn_uniform_normalized.pt")

bl_un_errors = bl.evaluate(model_bl_un, best_ll_bl, bl_ref, device)
print(f"rel-L2 per snapshot: {bl_un_errors}")
print(f"Mean: {np.mean(list(bl_un_errors.values())):.4f}")

## Summary (Apr 21 2026 — filled in)

| Claim | Pre-audit | Post-audit | Verdict |
|-------|-----------|------------|---------|
| BL: best λ_pde ≈ 0.001 | reported | fair sweep → BL uniform λ_pde=0.120, Burgers λ_pde=0.733 | **artefact** of biased metric |
| Burgers uniform: rel-L2 0.0004 | reported | fair sweep 0.0005, seed-stable at σ=0.0000 | **robust** |
| Uniform beats logspace on Burgers | 2× | fair 0.0005 vs 0.0012 (2.4×) | **robust, stronger under fair metric** |
| Burgers `uniform_normalized` rel-L2 | — | 0.0008 | **worse** than uniform, better than logspace |
| BL `uniform_normalized` rel-L2 | — | 0.2226 | **worst** of the three modes |
| λ-invariance (network robust to λ) | untested | envelope ≤10⁻³, rel-L2 std ≤4×10⁻⁴ over 200 λ | **strongly confirmed** → main paper finding |

Full tables with per-seed / per-model numbers recorded in `docs/EXPERIMENTS.md` under "§1 Results", "§2 Results", "§3 Results", "§4 Results", and "Audit summary table".